# Kiva Microfinance Loan Portfolio — Data Preparation

**Author:** Emmanuel Kironji  
**Dataset:** [Kiva Loans — Data Science for Good (Kaggle)](https://www.kaggle.com/datasets/kiva/data-science-for-good-kiva-crowdfunding)  
**Purpose:** Clean and engineer the raw Kiva loans dataset for import into Power BI, where a 3-page executive dashboard was built to analyse loan portfolio health, funding gaps, and borrower demographics.

---

## Overview

The raw dataset contains **671,205 loan records** across 54 columns spanning 2014–2017.  
This notebook performs the following steps:

1. **Column selection** — reduce to 15 relevant columns
2. **Null handling** — fill or drop missing values appropriately
3. **Feature engineering** — create `funding_gap`, `fully_funded`, `funding_status`, date parts, and `primary_gender`
4. **Data type enforcement** — ensure numeric columns are correctly typed
5. **Export** — save clean CSV for Power BI


## 1. Imports

In [1]:
import pandas as pd
import re

## 2. Load Raw Data

In [3]:
kiva_loans_data= r"C:\Users\Admin\Desktop\Data Science projects\kiva_loans.csv"
kiva_loans = pd.read_csv(kiva_loans_data)

print(f'Raw shape: {kiva_loans.shape}')
print(f'Columns: {list(kiva_loans.columns)}')

Raw shape: (671205, 20)
Columns: ['id', 'funded_amount', 'loan_amount', 'activity', 'sector', 'use', 'country_code', 'country', 'region', 'currency', 'partner_id', 'posted_time', 'disbursed_time', 'funded_time', 'term_in_months', 'lender_count', 'tags', 'borrower_genders', 'repayment_interval', 'date']


## 3. Column Selection

The raw dataset has 54 columns. We keep only the 15 that are directly relevant to portfolio analysis, funding metrics, and borrower demographics — reducing noise and improving Power BI performance.

In [16]:
cols_to_keep = [
    'id',               # Unique loan identifier
    'funded_amount',    # Amount actually funded by lenders
    'loan_amount',      # Amount requested by borrower
    'activity',         # Specific borrower activity (e.g. Fruit & Vegetables)
    'sector',           # Broad sector (e.g. Agriculture, Food, Retail)
    'country',          # Borrower country
    'region',           # Borrower region within country
    'currency',         # Local currency of loan
    'partner_id',       # Kiva field partner ID
    'posted_time',      # When the loan was posted on Kiva
    'funded_time',      # When the loan was fully funded (has nulls)
    'term_in_months',   # Repayment term in months
    'lender_count',     # Number of individual lenders who contributed
    'repayment_interval', # monthly / irregular / bullet / weekly
    'borrower_genders'  # Gender string(s) of borrower(s)
]

kiva_loans = kiva_loans[cols_to_keep]

print(f' shape: {kiva_loans.shape}')
print(f'\nNull counts:\n{kiva_loans.isnull().sum()}')

 shape: (671205, 15)

Null counts:
id                        0
funded_amount             0
loan_amount               0
activity                  0
sector                    0
country                   0
region                56800
currency                  0
partner_id            13507
posted_time               0
funded_time           48331
term_in_months            0
lender_count              0
repayment_interval        0
borrower_genders       4221
dtype: int64


## 4. Null Handling

| Column | Nulls | Action | Reason |
|---|---|---|---|
| `region` | 56,800 | Fill → `'Unknown'` | Not a primary analysis dimension; dropping rows is unnecessary |
| `partner_id` | 13,507 | Fill → `0` | Reference field; nulls don't affect aggregations |
| `funded_time` | 48,331 | Drop column | Not used in any dashboard visual |
| `borrower_genders` | 4,221 | Handled via classification | Nulls map to `'Unknown'` in the gender function |

In [17]:
# Fill region nulls
kiva_loans['region'] = kiva_loans['region'].fillna('Unknown')

# Fill partner_id nulls
kiva_loans['partner_id'] = kiva_loans['partner_id'].fillna(0).astype(int)

# Drop funded_time — too many nulls and not needed
kiva_loans.drop(columns=['funded_time'], inplace=True)

print('Null counts after handling:')
print(kiva_loans.isnull().sum())

Null counts after handling:
id                       0
funded_amount            0
loan_amount              0
activity                 0
sector                   0
country                  0
region                   0
currency                 0
partner_id               0
posted_time              0
term_in_months           0
lender_count             0
repayment_interval       0
borrower_genders      4221
dtype: int64


## 5. Feature Engineering

### 5.1 Funding Metrics

- **`funding_gap`** — difference between requested and funded amount. Measures unmet borrower demand.
- **`fully_funded`** — binary flag (1 = fully funded, 0 = not). Used for KPI aggregations in Power BI.
- **`funding_status`** — human-readable version of `fully_funded` for dashboard legend labels.

In [18]:
kiva_loans['funding_gap'] = kiva_loans['loan_amount'] - kiva_loans['funded_amount']
kiva_loans['fully_funded'] = (kiva_loans['funding_gap'] <= 0).astype(int)
kiva_loans['funding_status'] = kiva_loans['fully_funded'].map({
    1: 'Fully Funded',
    0: 'Not Fully Funded'
})

fully_funded_rate = round(kiva_loans['fully_funded'].mean() * 100, 2)
print(f'Fully funded rate: {fully_funded_rate}%')
print(kiva_loans['funding_status'].value_counts())

Fully funded rate: 92.8%
funding_status
Fully Funded        622877
Not Fully Funded     48328
Name: count, dtype: int64


### 5.2 Date Parts

Power BI requires separate date part columns to build time-series charts.  
`month_name` is sorted by `month` number so Power BI renders Jan → Dec in correct calendar order rather than alphabetically.

In [19]:
kiva_loans['posted_time'] = pd.to_datetime(kiva_loans['posted_time'])
kiva_loans['year'] = kiva_loans['posted_time'].dt.year
kiva_loans['month'] = kiva_loans['posted_time'].dt.month
kiva_loans['month_name'] = kiva_loans['posted_time'].dt.strftime('%b')

# Enforce correct calendar sort order
month_order = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
               'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
kiva_loans['month_name'] = pd.Categorical(
    kiva_loans['month_name'],
    categories=month_order,
    ordered=True
)

print('Year range:', kiva_loans['year'].min(), '–', kiva_loans['year'].max())
print('Month distribution:\n', kiva_loans['month_name'].value_counts().sort_index())

Year range: 2014 – 2017
Month distribution:
 month_name
Jan    53813
Feb    60268
Mar    69644
Apr    60962
May    67664
Jun    66291
Jul    51641
Aug    45879
Sep    48259
Oct    49085
Nov    50809
Dec    46890
Name: count, dtype: int64


### 5.3 Borrower Gender Classification

The `borrower_genders` column contains strings like `'female, female, male'` representing group loans.

A naive `str.count('male')` misclassifies borrowers because `'male'` is a substring of `'female'`.  
The fix uses **regex word boundary matching** (`\bfemale\b`, `\bmale\b`) to count exact word occurrences, then classifies each loan by majority gender.

In [20]:
def classify_gender(x):
    if pd.isnull(x):
        return 'Unknown'
    x = str(x).lower()
    females = len(re.findall(r'\bfemale\b', x))
    males = len(re.findall(r'\bmale\b', x))
    if females > males:
        return 'Female'
    elif males > females:
        return 'Male'
    elif females == males and females > 0:
        return 'Mixed / Equal'
    else:
        return 'Unknown'

kiva_loans['primary_gender'] = kiva_loans['borrower_genders'].apply(classify_gender)

print('Gender distribution:')
print(kiva_loans['primary_gender'].value_counts())
print(f'\nFemale share: {round(kiva_loans["primary_gender"].eq("Female").mean() * 100, 1)}%')

# Drop original column 
kiva_loans.drop(columns=['borrower_genders'], inplace=True)

Gender distribution:
primary_gender
Female           514797
Male             146568
Mixed / Equal      5619
Unknown            4221
Name: count, dtype: int64

Female share: 76.7%


## 6. Data Type Enforcement

Ensure all numeric columns are correctly typed before export.  
Power BI will throw a `SUM cannot work on String` DAX error if numeric columns are read as object dtype.

In [21]:
numeric_cols = [
    'funded_amount', 'loan_amount', 'funding_gap',
    'term_in_months', 'lender_count', 'fully_funded'
]

for col in numeric_cols:
    kiva_loans[col] = pd.to_numeric(kiva_loans[col], errors='coerce')

print('Final dtypes:')
print(kiva_loans.dtypes)

Final dtypes:
id                                  int64
funded_amount                     float64
loan_amount                       float64
activity                           object
sector                             object
country                            object
region                             object
currency                           object
partner_id                          int64
posted_time           datetime64[ns, UTC]
term_in_months                    float64
lender_count                        int64
repayment_interval                 object
funding_gap                       float64
fully_funded                        int64
funding_status                     object
year                                int32
month                               int32
month_name                       category
primary_gender                     object
dtype: object


## 7. Export for Power BI

In [4]:
kiva_loans.to_csv('kiva_loans_clean.csv', index=False)
print(f' Exported kiva_loans_clean.csv — shape: {kiva_loans.shape}')

 Exported kiva_loans_clean.csv — shape: (671205, 20)


---

## Summary of Engineered Columns

| Column | Type | Description |
|---|---|---|
| `funding_gap` | float | `loan_amount` − `funded_amount`. Measures unmet borrower demand. |
| `fully_funded` | int (0/1) | Binary flag: 1 if loan was fully funded, 0 otherwise. |
| `funding_status` | str | Text label: `'Fully Funded'` or `'Not Fully Funded'`. |
| `year` | int | Year extracted from `posted_time`. |
| `month` | int | Month number (1–12) extracted from `posted_time`. |
| `month_name` | Categorical | Month abbreviation (Jan–Dec), sorted in calendar order. |
| `primary_gender` | str | Majority gender of borrower(s): Female / Male / Mixed / Equal / Unknown. |



